# Train LEDGAR classifiers on Colab

This notebook trains Legal-BERT, BERT-base, Legal-BERT LoRA, LSTM, and LSTM + GloVe one after another. It starts with a Legal-BERT smoke run when enabled.

| Model | T4 | L4 | A100 |
| --- | --- | --- | --- |
| Legal-BERT | to be measured | to be measured | to be measured |
| BERT-base | to be measured | to be measured | to be measured |
| Legal-BERT LoRA | to be measured | to be measured | to be measured |
| LSTM | to be measured | to be measured | to be measured |
| LSTM + GloVe | to be measured | to be measured | to be measured |

Finished runs are saved to Google Drive, so a Colab disconnection does not lose their outputs. Rerun all cells to skip completed runs. Outside Colab, the notebook runs one small CPU smoke check in the current repository.

In [ ]:
REPO_URL = "https://github.com/Exowz/clausier-ledgar.git"
BRANCH = "main"
CONFIGS = [
    "configs/legal_bert.yaml",
    "configs/bert_base.yaml",
    "configs/legal_bert_lora.yaml",
    "configs/lstm.yaml",
    "configs/lstm_glove.yaml",
]  # Trained one after the other.
DRIVE_DIR = "/content/drive/MyDrive/clausier"
SMOKE_FIRST = True  # Run the first config with --smoke on the GPU before full training.
SKIP_DONE = True  # Skip runs that already have metrics.json after a disconnection.
EXTRA_ARGS = ""  # For example: "--epochs 3 --batch-size 16".


In [ ]:
import os
import sys
from pathlib import Path

ON_COLAB = "google.colab" in sys.modules
LOCAL_CHECK = not ON_COLAB or os.environ.get("CLAUSIER_NB_LOCAL") == "1"

if LOCAL_CHECK:
    start = Path(os.getcwd()).resolve()
    REPO_ROOT = next(
        (directory for directory in (start, *start.parents) if (directory / "src").is_dir()),
        None,
    )
    if REPO_ROOT is None:
        raise FileNotFoundError("Run this notebook from inside the Clausier repository.")
    os.chdir(REPO_ROOT)
    OUT_DIR = REPO_ROOT / "outputs" / "colab_check"
    print(f"Mode: local check (one CPU smoke run); repository: {REPO_ROOT}")
else:
    REPO_ROOT = Path("/content/clausier-ledgar")
    OUT_DIR = Path(DRIVE_DIR) / "outputs"
    print(f"Mode: Colab GPU; outputs: {OUT_DIR}")


In [ ]:
import subprocess

if LOCAL_CHECK:
    from src.utils import get_device

    print(f"Device: {get_device()}")
else:
    # nvidia-smi only: importing torch here would load numpy before the pinned install below.
    if subprocess.run(["nvidia-smi"]).returncode != 0:
        raise RuntimeError("No CUDA GPU found. Select Runtime > Change runtime type > GPU.")

In [ ]:
if not LOCAL_CHECK:
    from google.colab import drive

    drive.mount("/content/drive")
    if (REPO_ROOT / ".git").is_dir():
        subprocess.run(["git", "-C", str(REPO_ROOT), "checkout", BRANCH], check=True)
        subprocess.run(["git", "-C", str(REPO_ROOT), "pull", "--ff-only", "origin", BRANCH], check=True)
    else:
        subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, str(REPO_ROOT)], check=True)
    os.chdir(REPO_ROOT)
    commit = subprocess.run(
        ["git", "rev-parse", "--short", "HEAD"], check=True, capture_output=True, text=True,
    ).stdout.strip()
    print(f"Repository commit: {commit}")


In [ ]:
if not LOCAL_CHECK:
    subprocess.run(
        ["bash", "-c", "grep -v '^torch==' requirements.txt > /tmp/req-colab.txt && pip install -q -r /tmp/req-colab.txt"],
        check=True,
    )
    subprocess.run(
        [sys.executable, "-c",
         "import torch, transformers, datasets; "
         "print(f'torch {torch.__version__}, transformers {transformers.__version__}, '"
         "f'datasets {datasets.__version__}, GPU {torch.cuda.get_device_name(0)}')"],
        check=True,
    )

In [ ]:
import shlex
import yaml

OUT_DIR.mkdir(parents=True, exist_ok=True)
jobs = [(CONFIGS[0], True)] if LOCAL_CHECK or SMOKE_FIRST else []
if not LOCAL_CHECK:
    jobs.extend((config, False) for config in CONFIGS)

for cfg, smoke in jobs:
    with (REPO_ROOT / cfg).open(encoding="utf-8") as file:
        run_name = yaml.safe_load(file)["run_name"]
    if smoke:
        run_name += "_smoke"
    run_dir = OUT_DIR / run_name
    if SKIP_DONE and (run_dir / "metrics.json").is_file():
        print(f"Skipped {run_name}: metrics.json already exists.")
        continue

    run_dir.mkdir(parents=True, exist_ok=True)
    command = [
        sys.executable, "-m", "src.classification.train",
        "--config", cfg, "--output-dir", str(OUT_DIR),
        *shlex.split(EXTRA_ARGS),
    ]
    if smoke:
        command.append("--smoke")
    if LOCAL_CHECK:
        command.extend(["--device", "cpu", "--n-train", "256"])
    log_path = run_dir / "train.log"
    print(f"Training {run_name}; log: {log_path}", flush=True)
    env = {**os.environ, "PYTHONUNBUFFERED": "1"}
    with log_path.open("w", encoding="utf-8") as log, subprocess.Popen(
        command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env,
    ) as process:
        for line in process.stdout:
            print(line, end="", flush=True)
            log.write(line)
    if process.returncode != 0:
        raise RuntimeError(f"{run_name} failed (exit code {process.returncode}), see {log_path}")

In [ ]:
import json

from IPython.display import Image, display

columns = [
    "run", "best epoch", "val macro-F1", "test accuracy", "test micro-F1",
    "test macro-F1", "test_dedup macro-F1", "trainable parameters",
    "training memory estimate (GB)", "train time (min)",
]
rows = []
finished_runs = []
for run_dir in sorted(path for path in OUT_DIR.iterdir() if path.is_dir()):
    metrics_path = run_dir / "metrics.json"
    if not metrics_path.is_file():
        continue
    with metrics_path.open(encoding="utf-8") as file:
        metrics = json.load(file)
    with (run_dir / "config.json").open(encoding="utf-8") as file:
        config = json.load(file)
    rows.append({
        "run": run_dir.name,
        "best epoch": config.get("best_epoch"),
        "val macro-F1": metrics.get("validation", {}).get("macro_f1"),
        "test accuracy": metrics.get("test", {}).get("accuracy"),
        "test micro-F1": metrics.get("test", {}).get("micro_f1"),
        "test macro-F1": metrics.get("test", {}).get("macro_f1"),
        "test_dedup macro-F1": metrics.get("test_dedup", {}).get("macro_f1"),
        "trainable parameters": config.get("trainable_params"),
        "training memory estimate (GB)": config.get("train_memory_gb_estimate"),
        "train time (min)": config.get("train_time_s", 0) / 60,
    })
    finished_runs.append(run_dir)

def cell(value) -> str:
    if value is None:
        return ""
    return f"{value:.4f}" if isinstance(value, float) else str(value)

print(" | ".join(columns))
for row in rows:
    print(" | ".join(cell(row[column]) for column in columns))
for run_dir in finished_runs:
    curve_path = run_dir / "training_curves.png"
    if curve_path.is_file():
        print(run_dir.name)
        display(Image(filename=str(curve_path)))


In [ ]:
from zipfile import ZIP_DEFLATED, ZipFile

audit_files = (
    "config.json", "metrics.json", "history.json", "training_curves.png", "train.log",
    "val_probs.npy", "test_probs.npy", "val_labels.npy", "test_labels.npy",
)
for run_dir in finished_runs:
    missing = [name for name in audit_files if not (run_dir / name).is_file()]
    if missing:
        raise FileNotFoundError(f"{run_dir.name} is missing audit files: {missing}")
    archive_path = OUT_DIR / f"{run_dir.name}.zip"
    with ZipFile(archive_path, "w", compression=ZIP_DEFLATED) as archive:
        for name in audit_files:
            archive.write(run_dir / name, arcname=name)
    print(f"Packaged {archive_path}")
    print(f"Download and unzip into outputs/{run_dir.name}/ in your local clone.")
    print(f"Model weights remain at {run_dir / 'model'}")

if not LOCAL_CHECK:
    from google.colab import files

    def download_run(run_name: str) -> None:
        '''Download one packaged audit run from Google Drive.'''
        files.download(str(OUT_DIR / f"{run_name}.zip"))

    print("To download a zip now, call download_run('legal_bert') in a new cell.")
